In [ ]:
import numpy as np
import random

class QLDPCGenerator:
    """A class for generating different Density Parity-Check (QLDPC) codes."""
    
    def __init__(self):
        """
        Initialize the QLDPC generator.
        """
    
    def show_code_parameters(self):
        """
        Show all the information about a QLDPC code.
        
        Returns:
            dict(dictionary): A dictionary containing code parameters.
        """
    
    def show_parity_matrices(self):
        """
        Generate a QLDPC code with its parity matrices.
        
        Returns:
            Hx(nd.array): A parity matrix.
            Hz(nd.array): A parity matrix.
        """
    
class HGPGenerator(QLDPCGenerator):
    """A child class for generating HyperGraph Product Codes."""

    def __init__(self, n1, k1, H1, n2, k2, H2):
        """
        Initialize the HGP codes.
        
        Args:
            n1 (int): Length of the first classical code.
            k1 (int): Dimension of the first classical code.
            H1 (nd.array): The parity check matrix of the first classical code.
            n2 (int): Length of the second classical code.
            k2 (int): Dimension of the second classical code.
            H2 (nd.array): The parity check matrix of the second classical code.
        """
        self._code_parameter_check(n1, H1)
        self._code_parameter_check(n2, H2)
        self.n1 = n1
        self.k1 = k1
        self.H1 = H1
        self.n2 = n2
        self.k2 = k2
        self.H2 = H2
        self.r1, _ = H1.shape
        self.r2, _ = H2.shape
        #TODO: distance
        self.n, self.k = self._calculate_HGP_parameters(n1, k1, H1, n2, k2, H2)
    
    def _code_parameter_check(n, H):
        """
        Check the dimensions of the parameters.
        
        Raise Errors.
        """
        _, n1 = H.shape
        if n1 != n:
            raise Exception()
    
    def _calculate_HGP_parameters(self):
        """
        Calculate the HGP parameters [[n,k]].
        
        Returns:
            n (int): Length of the first classical code.
            k (int): Dimension of the first classical code.
        """
        k1T = rank(self.H1.T)
        k2T = rank(self.H2.T)
        n = self.n1*self.n2 + self.r1*self.r2
        k = self.k1*self.k2 + k1T*k2T
        return n, k
        
    def get_parity_matrices(self):
        """
        Get the parity matrices of HGP codes.
        
        Returns:
            Hx(nd.array): A parity matrix.
            Hz(nd.array): A parity matrix.
        """
        if self.Hx and self. Hz:
            return self.Hx, self.Hz
        
        I_n1 = np.eye(self.n1, dtype=int)
        I_n2 = np.eye(self.n2, dtype=int)
        I_r1 = np.eye(self.r1, dtype=int)
        I_r2 = np.eye(self.r2, dtype=int)
        
        self.Hx = np.block([np.kron(self.H1, I_n2), np.kron(I_r1, self.H2.T)])
        self.Hz = np.block([np.kron(I_n1. self.H2), np.kron(self.H1.T, I_r2)])
        return self.Hx, self.Hz


class FBGenerator(QLDPCGenerator):
    """A child class for generating Fiber Bundle Codes."""

    def __init__(self, n1, k1, H1, n2, k2, H2, G):
        """
        Initialize the FB codes.
        
        Args:
            n1 (int): Length of the first classical code.
            k1 (int): Dimension of the first classical code.
            H1 (nd.array): The parity check matrix of the first classical code.
            n2 (int): Length of the second classical code.
            k2 (int): Dimension of the second classical code.
            H2 (nd.array): The parity check matrix of the second classical code.
            G (List[nd.array]): The automorphism group G, requiring g \partial_f = \partial g f.
        """
        self._code_parameter_check(n1, H1)
        self._code_parameter_check(n2, H2)
        self.n1 = n1
        self.k1 = k1
        self.H1 = H1
        self.n2 = n2
        self.k2 = k2
        self.H2 = H2
        self.r1, _ = H1.shape
        self.r2, _ = H2.shape
        self.G = G
    
    def get_parity_matrices(self):
        """
        Get the parity matrices of FB Codes.
        
        Returns:
            Hx(nd.array): A parity matrix.
            Hz(nd.array): A parity matrix.
        """
        if self.Hx and self. Hz:
            return self.Hx, self.Hz
        
        I_n1 = np.eye(self.n1, dtype=int) #fiber vector space 1
        I_n2 = np.eye(self.n2, dtype=int) #fiber vector space 2
        I_r1 = np.eye(self.r1, dtype=int) #base vector space 1
        I_r2 = np.eye(self.r2, dtype=int) #base vector space 2

        Hz_part1 = np.zeros((self.r2, self.n1, self.r2, self.n1))
        #b2 \otimes f1
        for i in range(self.r2):
            for j in range(self.n1):
                Hz_part1[i,j,:,:] = np.kron(I_r2[i, :], I_n1[j, :])
        
        Hz_part2 = np.zeros((self.r1, self.n2, self.r1, self.n2))
        #b1 \otimes ZYY f2
        ZYY = self._generate_ZYY_mask_matrix()
        for i in range(self.r1):
            for j in range(self.n2):
                #TODO: let G can choose different group representation with different size
                Hz_part2[i,j,:,:] = np.kron(I_r1[i, :], self.G[ZYY[i,j]].matrix(self.n2)@I_n2[j, :])
        Hz_part2[0,j,:,:] = np.sum(Hz_part2)
        for i in range(1,self.r2):
            Hz_part2[i,j,:,:] = Hz_part2[0,j,:,:]
        self.Hz = np.block([Hz_part1, Hz_part2])
        
        Hx_part1 = np.zeros((self.r1, self.n1, self.r1, self.n1))
        #b1 \otimes f1
        for i in range(self.r1):
            for j in range(self.n1):
                Hx_part1[i,j,:,:] = np.kron(I_r1[i, :], I_n1[j, :])

        Hx_part2 = np.zeros((self.r1, self.n1, self.r1, self.n1))
        #b1 \otimes ZYY f1
        ZYY = self._generate_ZYY_mask_matrix()
        for i in range(self.r1):
            for j in range(self.n1):
                #TODO: let G can choose different group representation with different size
                Hx_part2[i,j,:,:] = np.kron(I_r1[i, :], self.G[ZYY[i,j]].matrix(self.n1)@I_n1[j, :])
        Hx_part2[0,j,:,:] = np.sum(Hx_part2)
        for i in range(1,self.r1):
            Hx_part2[i,j,:,:] = Hx_part2[0,j,:,:]
        self.Hx = np.block([Hx_part1, Hx_part2])

        return self.Hx, self.Hz
    
    def _generate_ZYY_mask_matrix(self):
        #spread the number of elements in G!
        ZYY = np.zeros(self.n2, self.r2)
        r = len(self.G)
        
        # Fill matrix by cycling through [1, 2, ..., r]
        for i in range(self.n2*self.r2):
            row = i // r
            col = i % r
            ZYY[row][col] = (i % r) + 1
        
        # Flatten matrix to a list for shuffling
        flat_matrix = [ZYY[i][j] for i in range(self.n2) for j in range(self.r2)]
        
        # Shuffle the flattened list
        random.shuffle(flat_matrix)
        
        # Reconstruct the matrix from the shuffled list
        for i in range(self.n2, self.r2):
            row = i // r
            col = i % r
            ZYY[row][col] = flat_matrix[i] * random.randint(0, 1) #random for 0 or \varphi_\tau in the paper
        return ZYY


class BPGenerator(QLDPCGenerator):
    """A child class for generating Balanced Product Codes."""

    def __init__(self, n1, k1, H1, n2, k2, H2, H):
        """
        Initialize the BP codes.
        
        Args:
            n1 (int): Length of the first classical code.
            k1 (int): Dimension of the first classical code.
            H1 (nd.array): The parity check matrix of the first classical code.
            n2 (int): Length of the second classical code.
            k2 (int): Dimension of the second classical code.
            H2 (nd.array): The parity check matrix of the second classical code.
            H (List[nd.array]): The group H is a common symmetry group of the two classical codes.
        """
        self._code_parameter_check(n1, H1)
        self._code_parameter_check(n2, H2)
        self.n1 = n1
        self.k1 = k1
        self.H1 = H1
        self.n2 = n2
        self.k2 = k2
        self.H2 = H2
        self.r1, _ = H1.shape
        self.r2, _ = H2.shape
        self.H = H
        self.Hinv = self._calculate_H_inv(self.H)
    
    def _calculate_H_inv(self, H):
        #TODO: calculate the inverse element of the H group.
        pass
    
    def get_parity_matrices(self):
        """
        Get the parity matrices of BP Codes.
        
        Returns:
            Hx(nd.array): A parity matrix.
            Hz(nd.array): A parity matrix.
        """
        if self.Hx and self. Hz:
            return self.Hx, self.Hz
        
        I_n1 = np.eye(self.n1, dtype=int) #C1
        I_n2 = np.eye(self.n2, dtype=int) #C1'
        I_r1 = np.eye(self.r1, dtype=int) #C2
        I_r2 = np.eye(self.r2, dtype=int) #C2'

        #Construct the vector spaces by cancelling out the symmetry H
        # C1 \otimes_H C1'
        L11 =  self._calculate_vector_spaces(self.n1, self.n2, I_n1, I_n2)
        # C2 \otimes_H C2'
        L22 =  self._calculate_vector_spaces(self.r1, self.r2, I_r1, I_r2)
        # C1 \otimes_H C2'
        L12 =  self._calculate_vector_spaces(self.n1, self.r2, I_n1, I_r2)
        # C2 \otimes_H C1'
        L21 =  self._calculate_vector_spaces(self.r1, self.n2, I_r1, I_n2)

        #Calculate the H matrix if it is a HGP code
        hgp = HGPGenerator(self.n1, self.k1, self.H1, self.n2, self.k2, self.H2)
        Hxprime, Hzprime = hgp.get_parity_matrices()

        #Construct the final H matrix
        Hxl = self._calculate_final_H_matrix_from_HGP_H_matrix(Hxprime[:, self.r1*self.n2:], I_n1, I_r1, I_n1, I_r2, L11, L22, L12, L21)
        Hxr = self._calculate_final_H_matrix_from_HGP_H_matrix(Hxprime[:, :self.r1*self.n2], I_n1, I_r1, I_n2, I_r1, L11, L22, L12, L21)
        Hzl = self._calculate_final_H_matrix_from_HGP_H_matrix(Hzprime[:, self.r1*self.n2:], I_n1, I_r1, I_n2, I_r2, L11, L22, L12, L21)
        Hzr = self._calculate_final_H_matrix_from_HGP_H_matrix(Hzprime[:, :self.r1*self.n2], I_n2, I_r1, I_n2, I_r2, L11, L22, L12, L21)
        return np.block([Hxl, Hxr]), np.block([Hzl, Hzr])
    
    def _calculate_vector_spaces(self, p: int, q:int, w: np.ndarray, v: np.ndarray):
        L = []
        for i in range(p):
            for j in range(q):
                for h, h_inv in zip(self.H, self.Hinv):
                    L.append(np.kron(w[i]@h, h_inv@v[j]))
        return L
    
    def _calculate_final_H_matrix_from_HGP_H_matrix(self, H: np.ndarray, w: np.ndarray, v: np.ndarray, x: np.ndarray, y: np.ndarray, L11, L22, L12, L21):
        Hout = np.zeros((self.n1*self.n2, self.r1*self.r2))
        for i in range(H.shape[0]):
            for j in range(H.shape[1]):
                if H[i,j] == 1:
                    m1, m2, m3, m4 = self._find_the_indices_from_old_indices(i,j)
                    for k, nk in enumerate(L11+L22):
                        for l, ml in enumerate(L12+L21):
                            if (np.kron(w[i]@h, h_inv@v[j]) == nk) and (np.kron(w[i]@h, h_inv@v[j]) == nk):
                                Hout[k,l] = 1
        return Hout

    
    def _find_the_indices_from_old_indices(self, i: int, j: int):
        m1 = 1
        m2 = 1
        m3 = 1
        m4 = 1
        return m1, m2, m3, m4

In [ ]:
r1= H1.shape[0]
n1= H1.shape[1]
r2= H2.shape[0]
n2= H2.shape[1]
I_n1 = np.eye(n1, dtype=int) #C1
I_n2 = np.eye(n2, dtype=int) #C1'
I_r1 = np.eye(r1, dtype=int) #C2
I_r2 = np.eye(r2, dtype=int) #C2'


In [ ]:
#####################修改_calculate_vector_spaces 并进行测试

In [2]:

import numpy as np

class MyClass:
    def __init__(self, H_matrices, Hinv_matrices):
        # H 和 Hinv 应该是一对一的，代表群元素及其逆对应的矩阵
        # 假设它们都是方阵，且维度与 w 和 v 的元素兼容
        self.H = H_matrices
        self.Hinv = Hinv_matrices

    def _calculate_vector_spaces(self, p: int, q: int, w: np.ndarray, v: np.ndarray):
        L_all = []
        all_krons_hashable = set() # 用于存储所有已添加的克罗内克积的哈希表示

        print(f"Starting _calculate_vector_spaces with p={p}, q={q}")
        print(f"w shape: {w.shape}, v shape: {v.shape}")
        print(f"H matrices count: {len(self.H)}")

        for i in range(p):
            for j in range(q):
                print(f"\n--- Processing (i={i}, j={j}) ---")
                
                # 计算当前的基础克罗内克积
                # 假设 w[i] 和 v[j] 是向量，或者可以进行克罗内克积的数组
                current_base_kron = np.kron(w[i], v[j])
                print(f"Base kron for (w[{i}], v[{j}]): \n{current_base_kron}")

                # 将NumPy数组转换为可哈希的形式进行检查
                current_base_kron_hashable = tuple(current_base_kron.flatten())

                if current_base_kron_hashable in all_krons_hashable:
                    print(f"  Base kron already exists in L_all. Skipping (i={i}, j={j}).")
                    continue # 跳过当前 (i, j) 组合的内层循环和 L 的创建

                print(f"  Base kron is new. Calculating L for (i={i}, j={j}).")
                L = [] # 为当前 (i, j) 组合创建一个新的空列表 L
                
                for h_idx, (h, h_inv) in enumerate(zip(self.H, self.Hinv)):
                    print(f"    Applying H[{h_idx}] and Hinv[{h_idx}]...")
                    
                    # 确保矩阵乘法维度兼容
                    # 假设 w[i] 是行向量 (1, D1) 或 D1维向量
                    # 假设 h 是 (D1, D1) 矩阵
                    # 假设 h_inv 是 (D2, D2) 矩阵
                    # 假设 v[j] 是列向量 (D2, 1) 或 D2维向量
                    
                    # 为了简化，我们假设 w[i] 和 v[j] 都是向量，并且 h, h_inv 是方阵
                    # 实际应用中需要确保维度匹配
                    try:
                        transformed_w = w[i] @ h
                        transformed_v = h_inv @ v[j]
                        result_kron = np.kron(transformed_w, transformed_v)
                        L.append(result_kron)
                        
                        # 将每个新生成的 result_kron 也添加到 all_krons_hashable 中
                        all_krons_hashable.add(tuple(result_kron.flatten()))
                        print(f"      Added transformed kron (H[{h_idx}]): \n{result_kron}")
                    except ValueError as e:
                        print(f"      Dimension mismatch for (w[{i}]@h) or (h_inv@v[{j}]): {e}")
                        # 可以选择跳过或处理错误

                L_all.append(L) # 将填充好的 L 列表添加到 L_all 中
                print(f"  Finished L for (i={i}, j={j}). L_all now has {len(L_all)} sub-lists.")

        return L_all

# --- 测试数据 ---

# 1. 定义 H 和 Hinv 矩阵
# 为了简单，使用2x2矩阵。它们可以是群的表示矩阵。
H_matrices = [
    np.array([[1, 0], [0, 1]]), # 身份元
    np.array([[0, 1], [1, 0]])  # 置换矩阵
]
Hinv_matrices = [
    np.array([[1, 0], [0, 1]]), # 身份元逆是自身
    np.array([[0, 1], [1, 0]])  # 置换矩阵逆是自身
]

# 2. 定义 w 和 v 数组
# 假设 w 和 v 的元素是1维向量，长度与 H 矩阵的维度兼容
w_data = np.array([
    [1, 0], # w[0]
    [1, 0], # w[1]
    [0, 1]  # w[2] - 故意与 w[0] 相同，用于测试去重
])

v_data = np.array([
    [1, 0], # v[0]
    [1, 0]  # v[1]
])

# 3. 创建 MyClass 实例
my_instance = MyClass(H_matrices, Hinv_matrices)

# 4. 调用函数
p_val = len(w_data)
q_val = len(v_data)

result_L_all = my_instance._calculate_vector_spaces(p_val, q_val, w_data, v_data)

# --- 打印结果 ---
print("\n--- Final Results (L_all) ---")
for idx, sublist in enumerate(result_L_all):
    print(f"Sublist {idx}:")
    for item in sublist:
        print(item)
    print("-" * 20)

print(f"\nTotal number of sub-lists in L_all: {len(result_L_all)}")

Starting _calculate_vector_spaces with p=3, q=2
w shape: (3, 2), v shape: (2, 2)
H matrices count: 2

--- Processing (i=0, j=0) ---
Base kron for (w[0], v[0]): 
[1 0 0 0]
  Base kron is new. Calculating L for (i=0, j=0).
    Applying H[0] and Hinv[0]...
      Added transformed kron (H[0]): 
[1 0 0 0]
    Applying H[1] and Hinv[1]...
      Added transformed kron (H[1]): 
[0 0 0 1]
  Finished L for (i=0, j=0). L_all now has 1 sub-lists.

--- Processing (i=0, j=1) ---
Base kron for (w[0], v[1]): 
[1 0 0 0]
  Base kron already exists in L_all. Skipping (i=0, j=1).

--- Processing (i=1, j=0) ---
Base kron for (w[1], v[0]): 
[1 0 0 0]
  Base kron already exists in L_all. Skipping (i=1, j=0).

--- Processing (i=1, j=1) ---
Base kron for (w[1], v[1]): 
[1 0 0 0]
  Base kron already exists in L_all. Skipping (i=1, j=1).

--- Processing (i=2, j=0) ---
Base kron for (w[2], v[0]): 
[0 0 1 0]
  Base kron is new. Calculating L for (i=2, j=0).
    Applying H[0] and Hinv[0]...
      Added transformed 

In [6]:
my_instance = MyClass(H_matrices, Hinv_matrices)
L11 =  my_instance._calculate_vector_spaces(self.n1, self.n2, I_n1, I_n2)
        # C2 \otimes_H C2'
L22 =  my_instance._calculate_vector_spaces(self.r1, self.r2, I_r1, I_r2)
        # C1 \otimes_H C2'
L12 =  my_instance._calculate_vector_spaces(self.n1, self.r2, I_n1, I_r2)
        # C2 \otimes_H C1'
L21 =  my_instance._calculate_vector_spaces(self.r1, self.n2, I_r1, I_n2)
n1n2 = len(L11)
r1r2 = len(L22)
n1r2= len(L12)
r1n2= len(L21)

[1 0]
[0 1]


In [ ]:
#############修改_calculate_final_H_matrix_from_HGP_H_matrix 

In [ ]:

def _calculate_final_Hxl_from_HGP_H_matrix(Hxprime[:, :n1*n2], I_n1, I_n2, I_r1, y: np.ndarray, L11,L21):
        Houtshape1 = n1n2
        l1 = n2
        Houtshape0 = r1n2
        l2 = n2
        w = I_n1
        v = I_n2
        x=I_r1
        y = I_n2
        Hout = np.zeros((Houtshape0, Houtshape1))
        for i in range(H.shape[0]):
            for j in range(H.shape[1]):
                if H[i,j] == 1:
                    m1 = j//l1
                    m2 = i//l2
                    m3 = j% l1
                    m4 = i%l2
                    current_kron_1 = np.kron(w[m1], v[m3])
                    current_kron_2 = np.kron(x[m2], y[m4])
                    if np.array_equal(current_kron_1, nk_array):
                            for l, ml_array in enumerate(L21): # 将 ml 改名为 ml_array
                                # 使用 np.array_equal() 来比较数组是否完全相等
                                if np.array_equal(current_kron_2, ml_array):
                                    Hout[k, l] = 1
        return Hxl


In [ ]:
#######_calculate_final_Hxl_from_HGP_H_matrix的测试

In [7]:
import numpy as np

# 假设这个函数可能属于某个类，或者是一个独立的工具函数
# 为了测试，我们先把它当作一个独立的函数

def _calculate_final_Hxl_from_HGP_H_matrix(
    Hxprime: np.ndarray, # 修正：Hxprime 作为完整的ndarray传入
    n1: int, n2: int, r1: int, # 修正：明确 n1, n2, r1 作为维度参数
    I_n1_basis: np.ndarray, # 修正：更清晰的参数名，表示基向量
    I_n2_basis: np.ndarray,
    I_r1_basis: np.ndarray,
    L11: list,
    L21: list
):
    # 修正变量名，使之与传入参数一致，并避免覆盖
    w_basis = I_n1_basis
    v_basis = I_n2_basis
    x_basis = I_r1_basis
    y_basis = I_n2_basis # 假设这里也是用 I_n2_basis

    # 修正 Hout 的形状计算
    Hout_shape_cols = n1 * n2 # 对应 Hxprime 的列数
    Hout_shape_rows = r1 * n2 # 对应 Hxprime 的行数（如果 Hxprime 是 (r1*n2, n1*n2)）

    # Hout 的实际维度应该与 L11 和 L21 的长度匹配，因为 k 和 l 是它们的索引
    # 假设 L11 的长度是 Hout 的行数，L21 的长度是 Hout 的列数
    Hout = np.zeros((len(L11), len(L21)), dtype=int)

    # 优化：将 L11 和 L21 转换为哈希映射，以便高效查找
    L11_map = {tuple(arr.flatten()): idx for idx, arr in enumerate(L11)}
    L21_map = {tuple(arr.flatten()): idx for idx, arr in enumerate(L21)}

    # 假设 H 是 Hxprime，或者 Hxprime 的一个切片
    # 这里我们使用 Hxprime 的整个矩阵进行迭代
    # 你的原代码是 for i in range(H.shape[0]): for j in range(H.shape[1]):
    # 假设 Hxprime 就是那个 H
    H_matrix_to_iterate = Hxprime # 或者 Hxprime[:, :n1*n2] 如果这是你的意图

    print(f"Building Hout matrix of shape ({len(L11)}, {len(L21)})")
    print(f"Iterating over H_matrix_to_iterate of shape {H_matrix_to_iterate.shape}")

    for i in range(H_matrix_to_iterate.shape[0]):
        for j in range(H_matrix_to_iterate.shape[1]):
            # 只有当 H_matrix_to_iterate[i,j] 为 1 时才进行后续计算
            if H_matrix_to_iterate[i, j] == 1:
                print(f"\nProcessing H_matrix_to_iterate[{i},{j}] == 1")
                
                # 修正 m1, m2, m3, m4 的计算，它们应该基于 n1, n2, r1
                # 假设 l1 = n2, l2 = n2 是正确的，并且它们是用于索引的步长
                # m1 = j // n2  # 对应 w_basis 的索引
                # m3 = j % n2   # 对应 v_basis 的索引
                # m2 = i // n2  # 对应 x_basis 的索引
                # m4 = i % n2   # 对应 y_basis 的索引 (如果 y_basis 也是 n2 维度)

                # 根据你的原代码，m1, m3 似乎与 j 相关，m2, m4 与 i 相关
                # 并且 l1, l2 都是 n2
                # 这意味着 Hxprime 的列索引 j 对应 (m1, m3) 组合，行索引 i 对应 (m2, m4) 组合
                m1 = j // n2
                m3 = j % n2
                m2 = i // n2
                m4 = i % n2

                # 确保索引不越界
                if not (0 <= m1 < w_basis.shape[0] and 0 <= m3 < v_basis.shape[0] and
                        0 <= m2 < x_basis.shape[0] and 0 <= m4 < y_basis.shape[0]):
                    print(f"  Index out of bounds for basis vectors. Skipping. m1={m1}, m3={m3}, m2={m2}, m4={m4}")
                    continue

                # 计算两个克罗内克积
                current_kron_1 = np.kron(w_basis[m1], v_basis[m3])
                current_kron_2 = np.kron(x_basis[m2], y_basis[m4])
                
                print(f"  Kron1 for (m1={m1}, m3={m3}): \n{current_kron_1}")
                print(f"  Kron2 for (m2={m2}, m4={m4}): \n{current_kron_2}")

                # 将当前克罗内克积转换为可哈希形式进行查找
                current_kron_1_hashable = tuple(current_kron_1.flatten())
                current_kron_2_hashable = tuple(current_kron_2.flatten())

                # 使用哈希映射进行查找，获取 L11 和 L21 中的索引
                k_idx = L11_map.get(current_kron_1_hashable)
                l_idx = L21_map.get(current_kron_2_hashable)

                if k_idx is not None and l_idx is not None:
                    Hout[k_idx, l_idx] = 1
                    print(f"  Match found! Setting Hout[{k_idx},{l_idx}] = 1")
                else:
                    print("  No match found in L11_map or L21_map for current krones.")
    
    # 你的原代码返回 Hx，但函数名是 _calculate_final_Hxl_from_HGP_H_matrix
    # 并且你计算的是 Hout。假设你最终想返回 Hout。
    return Hout

In [8]:
# --- 测试数据 ---

# 1. 定义维度参数
n1_val = 2
n2_val = 2
r1_val = 2 # 假设 r1 也是 2

# 2. 定义基向量（例如，单位矩阵的行向量）
# I_n1_basis 应该有 n1 个向量，每个向量的维度与 H 矩阵的兼容
I_n1_basis_data = np.array([
    [1, 0], # 对应 w_basis[0]
    [0, 1]  # 对应 w_basis[1]
])

# I_n2_basis 应该有 n2 个向量
I_n2_basis_data = np.array([
    [1, 0], # 对应 v_basis[0]
    [0, 1]  # 对应 v_basis[1]
])

# I_r1_basis 应该有 r1 个向量
I_r1_basis_data = np.array([
    [1, 0], # 对应 x_basis[0]
    [0, 1]  # 对应 x_basis[1]
])

# 3. 定义 Hxprime 矩阵
# Hxprime 的维度应该是 (r1*n2, n1*n2)
# 例如，如果 r1=2, n2=2, n1=2，那么 Hxprime 应该是 (4, 4)
# 我们手动设置一些 1，来触发内部逻辑
Hxprime_test = np.zeros((r1_val * n2_val, n1_val * n2_val), dtype=int)
Hxprime_test[0, 0] = 1 # 对应 (m2=0, m4=0) 和 (m1=0, m3=0)
Hxprime_test[1, 2] = 1 # 对应 (m2=0, m4=1) 和 (m1=1, m3=0)
Hxprime_test[2, 1] = 1 # 对应 (m2=1, m4=0) 和 (m1=0, m3=1)
print("Hxprime_test matrix:")
print(Hxprime_test)

# 4. 定义 L11 和 L21 列表
# 这些列表应该包含 Hout 矩阵的行和列所代表的克罗内克积
# 确保它们包含一些我们期望在 Hxprime 中找到的组合
L11_test = [
    np.kron(I_n1_basis_data[0], I_n2_basis_data[0]), # [1,0,0,0] (对应 m1=0, m3=0)
    np.kron(I_n1_basis_data[0], I_n2_basis_data[1]), # [0,1,0,0] (对应 m1=0, m3=1)
    np.kron(I_n1_basis_data[1], I_n2_basis_data[0]), # [0,0,1,0] (对应 m1=1, m3=0)
    np.array([9,9,9,9]) # 不匹配的示例
]

L21_test = [
    np.kron(I_r1_basis_data[0], I_n2_basis_data[0]), # [1,0,0,0] (对应 m2=0, m4=0)
    np.kron(I_r1_basis_data[0], I_n2_basis_data[1]), # [0,1,0,0] (对应 m2=0, m4=1)
    np.kron(I_r1_basis_data[1], I_n2_basis_data[0]), # [0,0,1,0] (对应 m2=1, m4=0)
    np.array([8,8,8,8]) # 不匹配的示例
]

# 5. 调用函数
final_Hout_matrix = _calculate_final_Hxl_from_HGP_H_matrix(
    Hxprime=Hxprime_test,
    n1=n1_val, n2=n2_val, r1=r1_val,
    I_n1_basis=I_n1_basis_data,
    I_n2_basis=I_n2_basis_data,
    I_r1_basis=I_r1_basis_data,
    L11=L11_test,
    L21=L21_test
)

# --- 打印最终 Hout 矩阵 ---
print("\n--- Final Hout Matrix ---")
print(final_Hout_matrix)

# 预期 Hout 矩阵的形状是 (len(L11_test), len(L21_test)) = (4, 4)

# 预期 Hout 矩阵的 1 应该在以下位置：
# Hxprime_test[0, 0] = 1:
#   m1=0, m3=0 => kron1 = L11_test[0] => k_idx = 0
#   m2=0, m4=0 => kron2 = L21_test[0] => l_idx = 0
#   => Hout[0,0] = 1

# Hxprime_test[1, 2] = 1:
#   m1=1, m3=0 => kron1 = L11_test[2] => k_idx = 2
#   m2=0, m4=1 => kron2 = L21_test[1] => l_idx = 1
#   => Hout[2,1] = 1

# Hxprime_test[2, 1] = 1:
#   m1=0, m3=1 => kron1 = L11_test[1] => k_idx = 1
#   m2=1, m4=0 => kron2 = L21_test[2] => l_idx = 2
#   => Hout[1,2] = 1

# 因此，最终 Hout 矩阵应该在 (0,0), (2,1), (1,2) 位置有 1。

Hxprime_test matrix:
[[1 0 0 0]
 [0 0 1 0]
 [0 1 0 0]
 [0 0 0 0]]
Building Hout matrix of shape (4, 4)
Iterating over H_matrix_to_iterate of shape (4, 4)

Processing H_matrix_to_iterate[0,0] == 1
  Kron1 for (m1=0, m3=0): 
[1 0 0 0]
  Kron2 for (m2=0, m4=0): 
[1 0 0 0]
  Match found! Setting Hout[0,0] = 1

Processing H_matrix_to_iterate[1,2] == 1
  Kron1 for (m1=1, m3=0): 
[0 0 1 0]
  Kron2 for (m2=0, m4=1): 
[0 1 0 0]
  Match found! Setting Hout[2,1] = 1

Processing H_matrix_to_iterate[2,1] == 1
  Kron1 for (m1=0, m3=1): 
[0 1 0 0]
  Kron2 for (m2=1, m4=0): 
[0 0 1 0]
  Match found! Setting Hout[1,2] = 1

--- Final Hout Matrix ---
[[1 0 0 0]
 [0 0 1 0]
 [0 1 0 0]
 [0 0 0 0]]


In [31]:
import numpy as np
def calculate_vector_spaces_with_h(p: int, q:int, w: np.ndarray, v: np.ndarray, H: list, Hinv: list):
    L = []
    L_index = 0
    Lh = []
    Lh_index = 0
    Ldict = {}
    for i in range(p):
        for j in range(q):
            L.append(np.kron(w[i], v[j]))
            for h, h_inv in zip(H, Hinv):
                current_kron = np.kron(w[i]@h, h_inv@v[j])
                matching_index = next((idx for idx, lh in enumerate(Lh) if np.allclose(current_kron, lh)), None)
                if matching_index is not None:
                    Ldict[matching_index].append(L_index)
                else:
                    Lh.append(current_kron)
                    Ldict[Lh_index] = []
                    Ldict[Lh_index].append(L_index)
                    Lh_index += 1
            L_index += 1
    return L, Lh, Ldict

In [35]:
p, q = 2, 3
w = np.array([[1, 0], [0, 1]])  # 2x2 vectors
v = np.array([[1, 0], [0, 1],[0,0]])  # 2x2 vectors
H = [np.eye(2)]  # 2x2 identity matrix
Hinv = [np.eye(2)]  # Inverse of identity is identity
L, Lh, Ldict = calculate_vector_spaces_with_h(p, q, w, v, H, Hinv)

p, q = 3, 3
w = np.array([[1, 0], [0, 1], [0,0]])  # 2x2 vectors
v = np.array([[1, 0], [0, 1],[0,0]])  # 2x2 vectors
H = [np.eye(2)]  # 2x2 identity matrix
Hinv = [np.eye(2)]  # Inverse of identity is identity
L2, Lh2, Ldict2 = calculate_vector_spaces_with_h(p, q, w, v, H, Hinv)

In [36]:
L, Lh, Ldict, L2, Lh2, Ldict2

([array([1, 0, 0, 0]),
  array([0, 1, 0, 0]),
  array([0, 0, 0, 0]),
  array([0, 0, 1, 0]),
  array([0, 0, 0, 1]),
  array([0, 0, 0, 0])],
 [array([1., 0., 0., 0.]),
  array([0., 1., 0., 0.]),
  array([0., 0., 0., 0.]),
  array([0., 0., 1., 0.]),
  array([0., 0., 0., 1.])],
 {0: [0], 1: [1], 2: [2, 5], 3: [3], 4: [4]},
 [array([1, 0, 0, 0]),
  array([0, 1, 0, 0]),
  array([0, 0, 0, 0]),
  array([0, 0, 1, 0]),
  array([0, 0, 0, 1]),
  array([0, 0, 0, 0]),
  array([0, 0, 0, 0]),
  array([0, 0, 0, 0]),
  array([0, 0, 0, 0])],
 [array([1., 0., 0., 0.]),
  array([0., 1., 0., 0.]),
  array([0., 0., 0., 0.]),
  array([0., 0., 1., 0.]),
  array([0., 0., 0., 1.])],
 {0: [0], 1: [1], 2: [2, 5, 6, 7, 8], 3: [3], 4: [4]})

In [55]:
HGPh = np.zeros((len(L),len(L2)))
HGPh[0,2] = 1
HGPh[0,5] = 1
HGPh[0,6] = 1
HGPh[0,7] = 1
HGPh[0,8] = 1
HGPh[2,2] = 1
HGPh[5,2] = 1
HGPh[0,3] = 1
HGPh

array([[0., 0., 1., 1., 0., 1., 1., 1., 1.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 1., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 1., 0., 0., 0., 0., 0., 0.]])

In [56]:
def calculate_final_H_matrix_from_HGP_H_matrix(HGPH: np.ndarray, L1H, L1dict, L2H, L2dict):
    Hout = np.zeros((len(L1H),len(L2H)))
    for i in range(HGPH.shape[0]):
        for j in range(HGPH.shape[1]):
            if HGPH[i,j] == 1:
                L1Hindices = corresponding_key(i, L1dict)
                L2Hindices = corresponding_key(j, L2dict)
                print(i,j,L1Hindices, L2Hindices)
                Hout[L1Hindices,L2Hindices] = 1
    return Hout
def corresponding_key(val, dictionary):
        for k, v in dictionary.items():
            if val in v:
                return k

In [57]:
calculate_final_H_matrix_from_HGP_H_matrix(HGPh, Lh, Ldict, Lh2, Ldict2)

0 2 0 2
0 3 0 3
0 5 0 2
0 6 0 2
0 7 0 2
0 8 0 2
2 2 2 2
5 2 2 2


array([[0., 0., 1., 1., 0.],
       [0., 0., 0., 0., 0.],
       [0., 0., 1., 0., 0.],
       [0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.]])